# Task 1: Cumulative Sum With `torch.nn.RNN`

In this task, the input is a sequence of digits and the target at each time step is the cumulative sum up to that position.

## TODO

Implement `RNNModel` using only PyTorch `torch.nn` modules.

- Use `nn.Embedding` to convert integer tokens from shape `(B, T)` to `(B, T, d_model)`.
- Use `nn.RNN` with `batch_first=True` so the recurrent output stays `(B, T, d_model)`.
- Use `nn.Linear` to project each time-step hidden state to `output_vocab` logits.
- Return logits with shape `(B, T, output_vocab)`.
- Do not apply `softmax`; `nn.CrossEntropyLoss` expects raw logits.

The maximum target value is `9 * seq_len`, so with `seq_len=25`, `output_vocab` should cover labels `0..225`.

In [1]:
import random
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

device = 'cuda' if torch.cuda.is_available() else 'cpu'
set_seed(217)
device

'cpu'

In [2]:
class CustomDataset(Dataset):
    def __init__(self, vocab_size=10, seq_len=25, size=40000):
        self.vocab_size = vocab_size
        self.seq_len = seq_len
        self.size = size
        self.X = torch.randint(0, vocab_size, (size, seq_len))
        self.Y = torch.cumsum(self.X, dim=1)

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

## Model TODO

Fill in the missing parts below. The intended flow is:

`tokens -> embedding -> nn.RNN -> linear classifier -> logits`

In [11]:
class RNNModel(nn.Module):
    def __init__(self, d_model=64, num_layers=2, seq_len=25):
        super().__init__()
        self.input_vocab = 10
        self.output_vocab = 9 * seq_len + 1

        # TODO: create nn.Embedding(self.input_vocab, d_model)
        self.embedding=nn.Embedding(self.input_vocab,d_model)
        # TODO: create nn.RNN(input_size=d_model, hidden_size=d_model,
        #                     num_layers=num_layers, batch_first=True)
        self.rnn=nn.RNN(input_size=d_model,hidden_size=d_model,num_layers=num_layers,batch_first=True)
        # TODO: create nn.Linear(d_model, self.output_vocab)
        self.fc=nn.Linear(d_model,self.output_vocab)
        return
        raise NotImplementedError('Define embedding, nn.RNN, and output projection')

    def forward(self, x):
        # x: (B, T) integer tokens
        emb=self.embedding(x)
        # TODO: embed x -> (B, T, d_model)
        out,_=self.rnn(emb)
        # TODO: run embedded sequence through nn.RNN
        # TODO: project every time step to output_vocab logits
        logits=self.fc(out)
        # TODO: return logits of shape (B, T, output_vocab)
        return logits
        raise NotImplementedError('Implement the forward pass with nn.RNN')

In [12]:
def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x).argmax(dim=-1)
            correct += (preds == y).sum().item()
            total += y.numel()
    return correct / total

def train(epochs=10, batch_size=256):
    model = RNNModel().to(device)
    dataset = CustomDataset()
    train_size = int(0.8 * len(dataset))
    test_size = len(dataset) - train_size
    train_set, test_set = random_split(dataset, [train_size, test_size])
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_set, batch_size=64)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
    criterion = nn.CrossEntropyLoss()

    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = criterion(logits.view(-1, logits.size(-1)), y.view(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'Epoch {epoch + 1}: loss={total_loss / len(train_loader):.4f}, acc={evaluate(model, test_loader, device) * 100:.2f}%')
    return model

# Run after completing RNNModel:
model = train()

Epoch 1: loss=3.2860, acc=18.34%
Epoch 2: loss=1.9662, acc=35.93%
Epoch 3: loss=1.5746, acc=27.90%
Epoch 4: loss=1.3742, acc=62.15%
Epoch 5: loss=1.2245, acc=42.82%
Epoch 6: loss=1.1175, acc=70.30%
Epoch 7: loss=1.0412, acc=50.77%
Epoch 8: loss=0.9775, acc=43.66%
Epoch 9: loss=0.9153, acc=81.60%
Epoch 10: loss=0.9046, acc=70.69%
